# Bigram vs LSTM vs Small Transformer — Next-Character Prediction

**Applied NLP — Midterm Project** · Aashay Gokhale

All three models solve the same task on Tiny Shakespeare: given the preceding
characters, predict the next one. The code lives in plain Python files —
`data.py`, `models.py`, `train.py` — and this notebook runs them in order and
shows the results. Explanations of each model are in the README.

**To run on Colab:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all.

In [ ]:
import os, sys
if not os.path.exists("train.py"):           # bare Colab session: fetch the repo
    !git clone -q https://github.com/Aashay1-ops/nlp-midterm.git
    %cd nlp-midterm
sys.path.insert(0, os.getcwd())

# QUICK = True runs a few steps per model to check the notebook end to end.
# The committed results come from QUICK = False.
QUICK = False

In [ ]:
import math
import torch
import matplotlib.pyplot as plt

import data
from data import decode, device, stoi, itos, vocab_size
from models import bigram_count_loss, count_parameters
from train import CONFIG, DISPLAY, generate, run, save_results, plot_curves

cfg = dict(CONFIG)
if QUICK:
    cfg.update(max_steps=60, warmup_steps=6, eval_interval=30)

print("device:", device, f"({torch.cuda.get_device_name(0)})" if device == "cuda" else "")
print("torch:", torch.__version__)
print({k: cfg[k] for k in ("block_size", "batch_size", "max_steps", "lr")})

## 1. Dataset and setup

Tiny Shakespeare, split contiguously: the first 90% for training, the last 10%
for validation. A random split would leak, because neighbouring training
windows overlap almost entirely.

In [ ]:
base = data.baselines()
counts = bigram_count_loss(data.train_data, data.val_data, vocab_size)

print(f"characters:       {len(data.text):,}")
print(f"vocabulary:       {vocab_size}  {''.join(data.chars)!r}")
print(f"train / val:      {len(data.train_data):,} / {len(data.val_data):,}")
print(f"val chars unseen in train: {len(set(data.val_data.tolist()) - set(data.train_data.tolist()))}")
print()
print(f"uniform-guess loss  ln({vocab_size}) = {base['uniform']:.4f} nats/char")
print(f"unigram loss (frequencies only) = {base['unigram']:.4f} nats/char")

The two baselines say what a loss number means. A model that cannot beat the
unigram loss is not using context at all.

## 2. Model 1 — Bigram

One 65×65 table: row *i* is the logits for the character after character *i*.
It is trained by the same loop as the other two models, so its loss and time
are measured the same way.

### 2.1 How sparse is the table?

In [ ]:
print(f"possible pairs:              {vocab_size**2:,}")
print(f"pairs never seen in training: {counts['unseen_pairs']:,}")
print(f"val positions using an unseen pair: {counts['val_positions_with_unseen_pair']}")
print(f"closed-form add-1 bigram val loss:  {counts['val_loss']:.4f}  (best a bigram can do)")

### 2.2 Training

In [ ]:
results, models = {}, {}
models["bigram"], results["bigram"] = run("bigram", cfg)

### 2.3 What the table learned

The whole bigram model is this table, so it can be read directly.

In [ ]:
probs = torch.softmax(models["bigram"].table.weight.detach().cpu(), dim=-1)

def top_next(ch, k=5):
    p, i = probs[stoi[ch]].topk(k)
    return "  ".join(f"{itos[j]!r}:{q:.2f}" for q, j in zip(p.tolist(), i.tolist()))

for ch in ["q", "t", "h", " ", "\n", ":"]:
    print(f"after {ch!r:5}-> {top_next(ch)}")

### 2.4 Generated sample

In [ ]:
for prompt, text in results["bigram"]["samples"].items():
    print(f"--- prompt {prompt!r} ---\n{text}\n")

**Observations.** > ✏️ **Your turn** — write this after reading the output above. Delete this line when done.

## 3. Model 2 — LSTM

Embedding → 2-layer LSTM → linear head. Hidden size 358 was chosen so the LSTM
has the same number of parameters as the transformer (within 0.03%).

In [ ]:
models["lstm"], results["lstm"] = run("lstm", cfg)

### 3.1 Generated sample — same prompts, seed and temperature as the bigram

In [ ]:
for prompt, text in results["lstm"]["samples"].items():
    print(f"--- prompt {prompt!r} ---\n{text}\n")

**Observations — compared with the bigram.** > ✏️ **Your turn** — write this after reading the output above. Delete this line when done.

## 4. Model 3 — Small Transformer

| Component | Code |
| --- | --- |
| Token embeddings | `TransformerLM.tok_emb` |
| Positional embeddings (learned) | `TransformerLM.pos_emb` |
| Causal multi-head self-attention | `CausalSelfAttention` |
| Feedforward layer | `FeedForward` |
| Output layer | `TransformerLM.ln_f` + `head` |

4 pre-norm blocks, 6 heads, embedding size 192, context 256.

### 4.1 The causal mask is tested, not assumed

If the mask leaked, the model could read the answer, and the loss would look
great while the model learned nothing. The tests change a later character and
check that earlier predictions stay exactly the same.

In [ ]:
!{sys.executable} -m pytest -q test_models.py

### 4.2 Training — identical loop, batches, schedule and step budget to the LSTM

In [ ]:
models["transformer"], results["transformer"] = run("transformer", cfg)

### 4.3 Generated sample

In [ ]:
for prompt, text in results["transformer"]["samples"].items():
    print(f"--- prompt {prompt!r} ---\n{text}\n")

**Observations — compared with the LSTM.** > ✏️ **Your turn** — write this after reading the output above. Delete this line when done.

### 4.4 What the attention heads look at

Attention weights for every head in the first and last block, on the first 64
characters of the validation set. Rows are the position being predicted from,
columns are the earlier positions it reads, and the axes are labelled with the
actual characters, so you can see *what* a head attends to and not just where.

In [ ]:
tf = models["transformer"]
n_show = 64
x = data.val_data[:n_show].unsqueeze(0).to(device)
labels = [c if c not in " \n" else {" ": "␣", "\n": "↵"}[c] for c in decode(x[0].tolist())]

layers = [0, len(tf.blocks) - 1]
for b in tf.blocks:
    b.attn.keep_weights = True
tf.eval()
with torch.no_grad():
    tf(x)

n_head = cfg["n_head"]
fig, axes = plt.subplots(len(layers), n_head, figsize=(3.2 * n_head, 3.4 * len(layers)))
for r, layer in enumerate(layers):
    w = tf.blocks[layer].attn.last_weights[0].cpu()
    for h in range(n_head):
        ax = axes[r, h]
        ax.imshow(w[h], cmap="viridis", vmin=0, vmax=w[h].max())
        ax.set_title(f"block {layer} · head {h}", fontsize=9)
        ax.set_xticks(range(n_show)); ax.set_xticklabels(labels, fontsize=4)
        ax.set_yticks(range(n_show)); ax.set_yticklabels(labels, fontsize=4)
for b in tf.blocks:
    b.attn.keep_weights = False
fig.tight_layout()
os.makedirs("results", exist_ok=True)
fig.savefig("results/attention.png", dpi=160)
plt.show()

above_diag = torch.triu(torch.ones(n_show, n_show, dtype=torch.bool), diagonal=1)
leak = max(b.attn.last_weights[0][:, above_diag].max().item() for b in tf.blocks)
print(f"largest attention weight above the diagonal, any block: {leak:.3e}")
print("text shown:", repr(decode(x[0].tolist())))

**What the heads do.** > ✏️ **Your turn** — write this after reading the output above. Delete this line when done.

## 5. Results

### 5.1 Results table

In [ ]:
from IPython.display import Markdown, display
table = save_results(results, cfg)
display(Markdown(table))
print(f"baselines: uniform {base['uniform']:.4f} | unigram {base['unigram']:.4f} "
      f"| closed-form bigram {counts['val_loss']:.4f}")

Losses are nats per character over **every** character of each split (not a
sample). *Best Val* is the lowest validation loss at any checkpoint, then
re-scored on the full validation set. The *Generated Text Quality* column is a
judgement, so fill it in by hand in `results/results_table.md` and in the README.

### 5.2 Learning curves

In [ ]:
fig = plot_curves(results, "results/learning_curves.png", base)
plt.show()

In [ ]:
print(f"{'step':>6}" + "".join(f"{DISPLAY[n]:>14}" for n in ("lstm", "transformer")))
h_l, h_t = results["lstm"]["history"], results["transformer"]["history"]
for i, step in enumerate(h_l["step"]):
    print(f"{step:>6}{h_l['val'][i]:>14.4f}{h_t['val'][i]:>14.4f}")

### 5.3 Training speed

In [ ]:
for name, r in results.items():
    per_step = 1000 * r["train_time_s"] / cfg["max_steps"]
    print(f"{DISPLAY[name]:12} {r['train_time_s']:7.1f}s total | {per_step:6.2f} ms/step")

### 5.4 Samples side by side (prompt `ROMEO:`)

In [ ]:
for name, r in results.items():
    print("=" * 70)
    print(f"{DISPLAY[name]}  —  val loss {r['final']['val_loss']:.4f}")
    print("=" * 70)
    print(r["samples"]["ROMEO:\n"][:400], "\n")

### 5.5 Qualitative analysis

> ✏️ **Your turn** — write this after reading the output above. Delete this line when done. Answer each from **your** numbers and samples above.

**Which model performed best?**

**Which model trained fastest?**

**Which model generated the most realistic text?**

**What did the transformer improve compared to the other models?**

**Was the improvement worth the added complexity?**

## 6. Reflection

> ✏️ **Your turn** — write this after reading the output above. Delete this line when done.

**What I learned about attention.**

**What surprised me.**

**What I would do differently.**